In [35]:

# Reload both modules
import importlib
import Market_Expectation_Agent
import Stock_Trend_Read_Agent

importlib.reload(Market_Expectation_Agent)
importlib.reload(Stock_Trend_Read_Agent)

import Revenue_Segmentation_Read_Agent
importlib.reload(Revenue_Segmentation_Read_Agent)


import importlib
import Market_Expectation_Agent
importlib.reload(Market_Expectation_Agent)
# Test all three agents with your existing variables
import json
import asyncio
import yfinance as yf
from LLM_Call_Agent import LLMCallAgent
# pip install langchain-openai pydantic
from langchain_openai import ChatOpenAI
from langchain_core.pydantic_v1 import BaseModel, Field
from typing import List

import News_Verification
importlib.reload(News_Verification)

<module 'News_Verification' from '/Users/xikinki/Desktop/Fintegrate_AI_File/New_Fintegrate_AI(V2)/News_Verification.py'>

In [36]:
REDIS_CONFIG = {
    'host': 'redis-16376.crce197.us-east-2-1.ec2.redns.redis-cloud.com',
    'port': 16376,
    'password': 'rl8242B4UItBhFzgHW5APEqZnkYoaEZv'
}

In [37]:
user_question = "Federal Reserve Powell indicates conditions, 'may warrant' interest rate cuts as Fed proceeds 'carefully'"
user_id = "123"
ticker = "MSTR"

## Step 3). Quries to Impaction System

## Manager Agent 

In [3]:
class Manager_Agent_Result(BaseModel):
    Decision_call_market_expectation: int
    Decision_call_revenue_segmentation: int
    Decision_call_macro_analyst: int
    quer_for_market_expectation: str
    quer_for_revenue_segmentation: str
    quer_for_macro_analyst: str 


Manger_agent = LLMCallAgent(
    default_provider="deepseek",  # Your preferred default
    default_model="deepseek-chat"
)


# Get API keys from your LLM Call Agent (it already has them)
deepseek_api_key = Manger_agent.deepseek_api_key

# Now use those keys for LangChain if needed
llm = ChatOpenAI(
    api_key=deepseek_api_key,  # Use the key from your LLM Call Agent
    base_url="https://api.deepseek.com",
    model="deepseek-chat",
    temperature=0,
    timeout=60,
)

structured_llm = llm.with_structured_output(Manager_Agent_Result)

2025-08-23 18:16:24,121 - INFO - 📥 Using OpenAI API key from Stock_Trend_Storage_Agent
2025-08-23 18:16:24,123 - INFO - 📥 Using DeepSeek API key from Stock_Trend_Storage_Agent
2025-08-23 18:16:24,209 - INFO - ✅ OpenAI client initialized
2025-08-23 18:16:24,226 - INFO - ✅ DeepSeek client initialized
2025-08-23 18:16:24,227 - INFO - 🤖 LLM Call Agent initialized
2025-08-23 18:16:24,227 - INFO -    - Default provider: deepseek
2025-08-23 18:16:24,227 - INFO -    - Default model: deepseek-chat
2025-08-23 18:16:24,227 - INFO -    - OpenAI: Enabled
2025-08-23 18:16:24,228 - INFO -    - DeepSeek: Enabled


/Users/xikinki/anaconda3/envs/arviz_env/lib/python3.10/site-packages/langchain_openai/chat_models/base.py:1896: UserWarning: Received a Pydantic BaseModel V1 schema. This is not supported by method="json_schema". Please use method="function_calling" or specify schema via JSON Schema or Pydantic V2 BaseModel. Overriding to method="function_calling".
  warnings.warn(


In [5]:
# Add this after your structured_llm setup:
def process_manager_query(user_query: str, ticker: str) -> Manager_Agent_Result:
    """
    Process a user query and intelligently route to appropriate agents
    """
    
    # Your intelligent routing prompt
    prompt = f"""
    You are a Manager Agent that analyzes user queries and decides which specialized agents to call.
    
    USER QUERY: "{user_query}"
    TICKER: {ticker}
    
    AVAILABLE AGENTS AND THEIR CAPABILITIES:
    
    1. MARKET EXPECTATION AGENT:
       - Database: Stock trend time intervals, price behavior, micro/macro events, timeline segmentation
       - Best for: Finding similar historical trends based on events, policies, earnings
       - Example queries: "Given tariff cuts, find similar timeline trends with similar macro/micro events"
       - Decision: Call if query involves market trends, price behavior, or historical pattern matching
    
    2. REVENUE SEGMENTATION AGENT:
       - Database: Revenue breakdown (GPU %, Data Center %, etc.), customer segments
       - Best for: Revenue impact analysis of corporate partnerships, service changes, market shifts
       - Example queries: "How will Microsoft partnership affect CRWV revenue segments?"
       - Decision: Call if query involves revenue drivers, partnerships, or business model changes
    
    3. MACRO ANALYST AGENT:
       - Database: Macroeconomic indicators, policy changes, economic environment data
       - Best for: Economic factors affecting stock performance, policy impacts
       - Example queries: "How do interest rates affect CRWV?"
       - Decision: Call if query involves economic environment, policies, or macro factors
    
    YOUR TASK:
    1. Analyze the user query to determine which agents are relevant
    2. Generate specific, targeted queries for each relevant agent
    3. Set decision flags (1=call, 0=don't call) for each agent
    4. Ensure queries are specific and actionable for each agent's database
    
    OUTPUT FORMAT:
    - Decision_call_market_expectation: 1 if query involves trends/patterns, 0 otherwise
    - Decision_call_fundamental_segmentation: 1 if query involves revenue/business model, 0 otherwise  
    - Decision_call_macro_analyst: 1 if query involves economic environment, 0 otherwise
    - Generate specific queries only for agents you decide to call (1)
    - For agents you don't call (0), use "N/A" as the query
    """
    
    # Call the structured LLM
    result = structured_llm.invoke(prompt)
    return result



## Manager Agent Calling 

In [6]:


result = process_manager_query(user_question, ticker)
        
print("🤖 Manager Agent Analysis Results:")
print("=" * 50)
print(f"User Query: {user_question}")
print(f"Ticker: {ticker}")
print("\n📊 Agent Routing Decisions:")
print(f"Market Expectation Agent: {'✅ CALL' if result.Decision_call_market_expectation else '❌ SKIP'}")
print(f"Revenue Segmentation Agent: {'✅ CALL' if result.Decision_call_revenue_segmentation else '❌ SKIP'}")
print(f"Macro Analyst Agent: {'✅ CALL' if result.Decision_call_macro_analyst else '❌ SKIP'}")
        
print("\n🔍 Generated Queries:")

if result.Decision_call_market_expectation:
    print(f"Market: {result.quer_for_market_expectation}")
if result.Decision_call_revenue_segmentation:
    print(f"Revenue: {result.quer_for_revenue_segmentation}")
if result.Decision_call_macro_analyst:
    print(f"Macro: {result.quer_for_macro_analyst}")
            


2025-08-23 18:16:33,875 - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
🤖 Manager Agent Analysis Results:
User Query: Federal Reserve Powell indicates conditions, 'may warrant' interest rate cuts as Fed proceeds 'carefully'
Ticker: MSTR

📊 Agent Routing Decisions:
Market Expectation Agent: ✅ CALL
Revenue Segmentation Agent: ❌ SKIP
Macro Analyst Agent: ✅ CALL

🔍 Generated Queries:
Market: Find historical stock price trends and patterns for MSTR during previous Federal Reserve interest rate cut cycles, including timeline segmentation around similar 'careful' policy shift announcements and market reactions
Macro: Analyze how Federal Reserve interest rate cut expectations and Powell's 'careful' policy approach typically affect technology stocks like MSTR, including historical macroeconomic impacts on similar companies during rate cut cycles


## Agent_Calling_Form

In [ ]:
Decision_call_market_expectation = result.Decision_call_market_expectation
Decision_call_revenue_segmentation = result.Decision_call_revenue_segmentation
Decision_call_revenue_segmentation = result.Decision_call_revenue_segmentation
Decision_call_macro_analyst = result.Decision_call_macro_analyst

Market_Expectation_Agent_Query = result.quer_for_market_expectation
Revenue_Segmentation_Query = result.quer_for_revenue_segmentation
Macro_Query = result.quer_for_macro_analyst



Decision_List = [Decision_call_market_expectation, Decision_call_revenue_segmentation, Decision_call_macro_analyst]
Agent_List = ["Market_Expectation_Agent", "Revenue_Segmentation_Agent", "Macro_Analyst_Agent"]
Query_List = [Market_Expectation_Agent_Query, Revenue_Segmentation_Query, Macro_Query]  # Add this!

Agents_Calling_Form = {}
for i in range(len(Decision_List)):
    if Decision_List[i] == 1:
        Agents_Calling_Form[Agent_List[i]] = 1
        Agents_Calling_Form[Agent_List[i] + "_Query"] = Query_List[i]  # Fix: Add assignment!

print(Agents_Calling_Form)

{'Market_Expectation_Agent': 1, 'Market_Expectation_Agent_Query': "Find historical stock price trends and patterns for MSTR during previous Federal Reserve interest rate cut cycles, including timeline segmentation around similar 'careful' policy shift announcements and market reactions", 'Macro_Analyst_Agent': 1, 'Macro_Analyst_Agent_Query': "Analyze how Federal Reserve interest rate cut expectations and Powell's 'careful' policy approach typically affect technology stocks like MSTR, including historical macroeconomic impacts on similar companies during rate cut cycles"}


## Sub Agent Calling 

In [8]:
# Add this in a NEW cell (Cell 8)
async def call_agents_dynamically():
    """
    Dynamically call agents based on decision flags and collect results
    """
    # Use the variables you already defined in Cell 7
    global Decision_call_market_expectation, Decision_call_revenue_segmentation, Decision_call_macro_analyst
    global Market_Expectation_Agent_Query, Revenue_Segmentation_Query, Macro_Query
    global REDIS_CONFIG, ticker
    
    # Decision and agent mapping
    global Decision_List 
    global Agent_List
    global Query_List
    
    # Initialize results dictionary
    Agents_Results = {}
    
    print("🚀 Starting Dynamic Agent Execution...")
    print("=" * 60)
    
    # Dynamically call agents based on decisions
    for i in range(len(Decision_List)):
        if Decision_List[i] == 1:
            agent_name = Agent_List[i]
            agent_query = Query_List[i]
            
            print(f"✅ Calling {agent_name}...")
            print(f"📝 Query: {agent_query[:100]}...")
            
            try:
                # Call appropriate agent based on name
                if agent_name == "Market_Expectation_Agent":
                    from Market_Expectation_Agent import MarketExpectationAgent
                    agent = MarketExpectationAgent(
                        redis_host=REDIS_CONFIG['host'],
                        redis_port=REDIS_CONFIG['port'],
                        redis_password=REDIS_CONFIG['password']
                    )
                    agent_result = await agent.process_query(agent_query, ticker)  # Keep await - this IS async
                    Agents_Results[f"{agent_name}_Result"] = agent_result.get('stock_read_result', 'No result')
                    agent.close()
                    
                elif agent_name == "Revenue_Segmentation_Agent":
                    from Revenue_Segmentation_Read_Agent import RevenueSegmentationAnalystAgent
                    agent = RevenueSegmentationAnalystAgent(
                        redis_host=REDIS_CONFIG['host'],
                        redis_port=REDIS_CONFIG['port'],
                        redis_password=REDIS_CONFIG['password']
                    )
                    agent_result = agent.process_natural_query(agent_query, ticker)  # Remove await - NOT async
                    Agents_Results[f"{agent_name}_Result"] = agent_result
                    agent.close()
                    
                elif agent_name == "Macro_Analyst_Agent":
                    from Macro_Analyst_Agent import MacroAnalystAgent
                    agent = MacroAnalystAgent(user_id="Fintegrate_AI_Test")
                    agent_result = agent.process_macro_query(agent_query)  # Remove await - NOT async
                    Agents_Results[f"{agent_name}_Result"] = agent_result.get('analysis', 'No result')
                    if hasattr(agent, 'redis_client') and agent.redis_client:
                        agent.redis_client.close()
                
                print(f"✅ {agent_name} completed successfully")
                
            except Exception as e:
                print(f"❌ Error in {agent_name}: {e}")
                Agents_Results[f"{agent_name}_Result"] = f"Error: {str(e)}"
        else:
            print(f"⏭️ Skipping {Agent_List[i]} (decision = 0)")
    
    print("\n" + "=" * 60)
    print("📊 Final Results Summary:")
    print("=" * 60)
    
    # Display results
    for key, value in Agents_Results.items():
        print(f"{key}: {str(value)[:200]}...")
    
    return Agents_Results

In [9]:
# Execute the dynamic agent calling pipeline
print("🚀 Starting Dynamic Agent Pipeline...")
print("=" * 60)

# Run the dynamic agent calling function
final_results = await call_agents_dynamically()

# Create structured output format
agents_result = ""
for key, value in final_results.items():
    agents_result += f"{{{key}: {str(value)[:100]}...}} "

print(f"\n🎯 Final Structured Output:")
print("=" * 60)
print(agents_result)

# Store results for further use
print(f"\n📊 Results Summary:")
print(f"Total agents called: {len(final_results)}")
print(f"Available results: {list(final_results.keys())}")


🚀 Starting Dynamic Agent Pipeline...
🚀 Starting Dynamic Agent Execution...
✅ Calling Market_Expectation_Agent...
📝 Query: Find historical stock price trends and patterns for MSTR during previous Federal Reserve interest ra...
2025-08-23 18:16:55,408 - INFO - ✅ Frontend Redis connected: redis-16204.fcrce180.us-east-1-1.ec2.redns.redis-cloud.com:16204
2025-08-23 18:16:55,409 - INFO - ✅ Stock trend Redis connected: redis-16376.crce197.us-east-2-1.ec2.redns.redis-cloud.com:16376
2025-08-23 18:16:55,410 - INFO - Attempting to connect to Redis...
2025-08-23 18:16:55,410 - INFO - Host: redis-16376.crce197.us-east-2-1.ec2.redns.redis-cloud.com
2025-08-23 18:16:55,411 - INFO - Port: 16376
2025-08-23 18:16:55,411 - INFO - Username: default
2025-08-23 18:16:55,411 - INFO - Testing connection with ping command...
2025-08-23 18:16:55,612 - INFO - ✓ Ping successful - Redis server is reachable
2025-08-23 18:16:55,612 - INFO - ✓ Successfully connected to Redis
2025-08-23 18:16:55,613 - INFO - 📥 Using 

/Users/xikinki/Desktop/Fintegrate_AI_File/Streamlit_APP_Test/Stock_Trend_Storage_Agent.py:335: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, progress=False)


2️⃣ Processing trend segments...
3️⃣ Fetching news data with multiprocessing...
🚀 Starting multiprocessing news fetch for 13 trends
📊 Using 10 processes
🔄 Process 8: Processing downtrend4
🔄 Process 9: Processing uptrend5
🔄 Process 10: Processing downtrend5
🔄 Process 3: Processing uptrend2
🔄 Process 4: Processing downtrend2
🔄 Process 1: Processing uptrend1
🔄 Process 7: Processing uptrend4
🔄 Process 5: Processing uptrend3
🔄 Process 2: Processing downtrend1
🔄 Process 6: Processing downtrend3
✅ Process 2: Completed downtrend1 with 10 news articles
🔄 Process 11: Processing uptrend6
✅ Process 1: Completed uptrend1 with 18 news articles
🔄 Process 12: Processing downtrend6
✅ Process 11: Completed uptrend6 with 12 news articles
🔄 Process 13: Processing uptrend7
✅ Process 7: Completed uptrend4 with 24 news articles
✅ Process 4: Completed downtrend2 with 24 news articles
✅ Process 13: Completed uptrend7 with 5 news articles
✅ Process 3: Completed uptrend2 with 24 news articles
✅ Process 6: Comple

In [10]:
print(final_results["Market_Expectation_Agent_Result"])

🆕 Fresh data downloaded for MSTR. **SIMILAR TREND MAPPING**: 
<Similar Trend Time: uptrend2 2025-03-10, 2025-03-25>
<Reason: because similar macro as Federal Reserve maintaining dovish stance with unchanged rates at 4.25%-4.5% for second consecutive meeting, creating favorable monetary conditions for growth stocks. Trump administration's deregulatory agenda and potential strategic crypto reserve policies generating sector tailwinds for cryptocurrency-related companies. AI market expansion with predictions of two more years of AI-powered bull market driving technology sector optimism., micro as MicroStrategy's announcement of proposed $500 million perpetual preferred stock offering specifically earmarked for bitcoin acquisition, reinforcing its position as largest corporate Bitcoin holder. Company's strategic pivot toward AI with rollout of generative AI tools for its business intelligence platform, expanding beyond pure Bitcoin exposure. Successful implementation of Strategy One platfo

In [12]:
print(final_results["Macro_Analyst_Agent_Result"])

**OPPORTUNITY**
• **FACT:** Strong economic growth supporting corporate earnings potential
  - **EVIDENCE:** Real GDP growth of 0.73% quarter-over-quarter (annualizing to ~3.0%)
  - **RESULT:** Robust economic expansion creates favorable revenue environment for technology companies like MSTR, supporting fundamentals amid monetary policy uncertainty

• **FACT:** Declining long-term interest rates improving financing conditions
  - **EVIDENCE:** 30-year mortgage rates down -4.1% and 15-year rates down -5.3% over three months
  - **RESULT:** Lower long-term rates reduce discount rates for future tech earnings and improve access to capital for growth investments and refinancing

• **FACT:** Resilient consumer spending supporting technology demand
  - **EVIDENCE:** Retail sales increased 0.71% over three months
  - **RESULT:** Strong consumer activity indicates sustained demand for technology products and services, benefiting tech sector revenue streams

**RISK**
• **FACT:** Persistent infl